# Homework 6

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

## Setup

In [2]:
DATA = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'

df = pd.read_csv(DATA).fillna(0)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train, df_val, df_test = [d.reset_index(drop=True) for d in (df_train, df_val, df_test)]

y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

for d in (df_train, df_val, df_test):
    del d['fuel_efficiency_mpg']

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))
X_test = dv.transform(df_test.to_dict(orient='records'))

X_train.shape

(6000, 16)

In [3]:
def rmse(y, y_pred):
    se = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)

## Question 1

`Which original feature is used for splitting the data?`

**Answer: `model_year`**

In [4]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

feats = dv.get_feature_names_out()
root = feats[dt.tree_.feature[0]]
print(f'{root}  (original column: {root.split("=")[0]}, threshold: {dt.tree_.threshold[0]:.1f})')

model_year  (original column: model_year, threshold: 1997.5)


## Question 2

`What's the RMSE of this model on the validation data?`

**Answer: 1.837** (closest option)

In [5]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

print('RMSE = %.4f' % rmse(y_val, rf.predict(X_val)))

RMSE = 1.8332


## Question 3

`Which value gives the lowest validation RMSE?`

**Answer: 150**

In [6]:
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    print('n_estimators=%3d  rmse=%.3f' % (n, rmse(y_val, rf.predict(X_val))))

n_estimators= 10  rmse=1.833
n_estimators= 50  rmse=1.774
n_estimators=100  rmse=1.770
n_estimators=150  rmse=1.769


## Question 4

`What's the best max_depth, using the mean RMSE?`

**Answer: 10**

In [7]:
for d in [10, 15, 20, 25]:
    scores = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        scores.append(rmse(y_val, rf.predict(X_val)))
    print('max_depth=%2d  mean_rmse=%.3f' % (d, np.mean(scores)))

max_depth=10  mean_rmse=1.754
max_depth=15  mean_rmse=1.784
max_depth=20  mean_rmse=1.787
max_depth=25  mean_rmse=1.786


## Question 5

`What's the most important feature?`

**Answer: `vehicle_weight`**

In [8]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

importance = pd.Series(rf.feature_importances_, index=feats)
importance.groupby(importance.index.str.split('=').str[0]).sum().sort_values(ascending=False)

model_year             0.296107
vehicle_weight         0.203313
fuel_type              0.193555
horsepower             0.078271
engine_displacement    0.061281
acceleration           0.054998
origin                 0.052103
drivetrain             0.033651
num_doors              0.019401
num_cylinders          0.007318
dtype: float64

## Question 6

`Which eta leads to the best RMSE score on the validation dataset?`

**Answer: 0.1**

In [9]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,

        'objective': 'reg:squarederror',
        'nthread': 8,

        'seed': 1,
        'verbosity': 1,
    }

    model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                      evals=[(dtrain, 'train'), (dval, 'val')],
                      verbose_eval=False)

    print('eta=%.1f  rmse=%.4f' % (eta, rmse(y_val, model.predict(dval))))

eta=0.3  rmse=1.8265
eta=0.1  rmse=1.7248
